In [1]:
!pip install -q -U transformers accelerate huggingface_hub fastapi uvicorn sentence-transformers pillow requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.


In [2]:
import os, re, sys, glob, json, time, shutil, subprocess, requests
from pathlib import Path

# ================= EDIT HERE =================
# SRC forms (HF id only is enough):
#   "org/name"                    auto-pick best GGUF (Q4_K_M first), else safetensors repo
#   "org/name:Q5_K_M"             force a quant
#   "org/name/some-file.gguf"     exact file
#   "/kaggle/input/x/model.gguf"  local file or folder
SRC = "nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB"
NON_GGUF = "convert"   # repo has no .gguf: "convert" = llama.cpp q8_0 (fast)  |  "transformers" = bnb-4bit (slow)
CTX = 16384            # LLM context tokens
THINK = False          # False = try to disable thinking tokens (faster)
# =============================================

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret (ok for public models):", type(e).__name__)

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip().splitlines()
GPUS = [[x.strip() for x in l.split(",")] for l in smi if l.strip()]
assert GPUS, "no GPU. Kaggle: Settings > Accelerator > GPU T4 x2"
NGPU = len(GPUS)
LLM_GPU = "0"
OCR_GPU = "1" if NGPU > 1 else "0"
SM = ";".join(sorted({g[3].replace(".", "") for g in GPUS}))
VRAM_GB = int(GPUS[0][2]) / 1024
OCR_SIDE = 1280 if NGPU > 1 else 1024
print(f"GPUs={NGPU} {[g[1] for g in GPUS]} sm={SM} vram0={VRAM_GB:.1f}GB  LLM->gpu{LLM_GPU}  OCR->gpu{OCR_GPU}")


def sh(cmd, log="notebook.log"):
    with open(log, "a") as f:
        f.write(f"\n$ {cmd}\n")
        f.flush()
        r = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}\n" + open(log).read()[-3000:])


GPUs=2 ['Tesla T4', 'Tesla T4'] sm=75 vram0=15.0GB  LLM->gpu0  OCR->gpu1


In [3]:
LLAMA = Path("/kaggle/working/llama.cpp")
CU12 = Path("/kaggle/working/cu12libs")
BIN = None


def ld_path(binpath=None):
    parts = [str(binpath.parent)] if binpath else []
    parts += [str(p) for p in CU12.glob("nvidia/*/lib")]
    parts += ["/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib"]
    parts += [os.environ.get("LD_LIBRARY_PATH", "")]
    return ":".join(x for x in parts if x)


def gpu_ok(binpath):
    try:
        env = {**os.environ, "LD_LIBRARY_PATH": ld_path(binpath)}
        r = subprocess.run([str(binpath), "--list-devices"], capture_output=True, text=True, timeout=60, env=env)
        out = r.stdout + r.stderr
        print("  devices:", out.strip()[-300:].replace("\n", " | "))
        return "CUDA" in out
    except Exception as e:
        print("  bin check failed:", type(e).__name__, e)
        return False


def ensure_cu12():
    # prebuilt binary is CUDA 12.8. Kaggle has CUDA 13 toolkit. Pull cu12 runtime libs from pip into a private dir.
    if not list(CU12.glob("nvidia/cuda_runtime/lib/libcudart.so.12*")):
        sh(f"{sys.executable} -m pip install -q --no-deps --target {CU12} nvidia-cuda-runtime-cu12 nvidia-cublas-cu12")


def try_prebuilt():
    ensure_cu12()
    dest = Path("/kaggle/working/llama-prebuilt")
    found = list(dest.rglob("llama-server")) if dest.exists() else []
    if not found:
        rel = requests.get("https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest", timeout=30).json()
        asset = next(a for a in rel["assets"] if "cuda-12.8-amd64" in a["name"] and a["name"].endswith(".tar.gz"))
        dest.mkdir(exist_ok=True)
        sh(f"curl -sL {asset['browser_download_url']} -o /kaggle/working/llama-prebuilt.tgz && tar -xzf /kaggle/working/llama-prebuilt.tgz -C {dest} && chmod -R +x {dest}")
        found = list(dest.rglob("llama-server"))
    return found[0] if found else None


def cuda_stub():
    pats = ["/usr/local/cuda*/targets/*/lib/stubs/libcuda.so",
            "/usr/local/cuda*/lib64/stubs/libcuda.so",
            "/usr/lib/x86_64-linux-gnu/libcuda.so",
            "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
            "/usr/local/nvidia/lib64/libcuda.so.1",
            "/usr/lib64/libcuda.so.1"]
    for p in pats:
        hit = sorted(glob.glob(p))
        if hit:
            return hit[0]
    r = subprocess.run(r"find / \( -path /proc -o -path /sys \) -prune -o -name 'libcuda.so*' -print 2>/dev/null | head -5",
                       shell=True, capture_output=True, text=True)
    hits = [l for l in r.stdout.split() if l]
    return hits[0] if hits else None


def build_from_source():
    print("building from source for sm", SM, "(~10 min, estimate)")
    if not LLAMA.exists():
        sh(f"git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA}")
    stub = cuda_stub()
    print("libcuda:", stub)
    assert stub, "libcuda.so not found anywhere. cannot link CUDA driver. check: find / -name 'libcuda.so*'"
    stubdir = Path("/kaggle/working/stubs")
    stubdir.mkdir(exist_ok=True)
    link = stubdir / "libcuda.so"
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(stub)
    shutil.rmtree(LLAMA / "build", ignore_errors=True)
    sh(f'cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="{SM}" '
       f'-DCUDA_CUDA_LIBRARY={link} -DCMAKE_LIBRARY_PATH={stubdir} '
       f'-DLLAMA_CURL=OFF -DGGML_CCACHE=OFF -DLLAMA_BUILD_TESTS=OFF -DCMAKE_BUILD_TYPE=Release')
    sh(f"cmake --build {LLAMA}/build --target llama-server -j $(nproc)")
    return LLAMA / "build/bin/llama-server"


# 1) already here?  2) prebuilt CUDA12 tarball + pip cu12 libs (seconds)  3) build from source
cached = list(Path("/kaggle/working/llama-prebuilt").rglob("llama-server")) + list(LLAMA.glob("build/bin/llama-server"))
for c in cached:
    if gpu_ok(c):
        BIN = c
        break

if BIN is None:
    try:
        print("trying prebuilt llama-server + cu12 libs...")
        b = try_prebuilt()
        if b and gpu_ok(b):
            BIN = b
    except Exception as e:
        print("prebuilt failed:", type(e).__name__, str(e)[:300])

if BIN is None:
    BIN = build_from_source()
    assert gpu_ok(BIN), "built binary sees no CUDA. check notebook.log"

print("llama-server:", BIN)


trying prebuilt llama-server + cu12 libs...
  devices: Available devices: |   CUDA0: Tesla T4 (14911 MiB, 14806 MiB free) |   CUDA1: Tesla T4 (14911 MiB, 14806 MiB free)
llama-server: /kaggle/working/llama-prebuilt/cuda-12.8/llama-server


In [4]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

QUANT_PREF = ["Q4_K_M", "Q4_K_S", "IQ4_XS", "Q4_0", "Q5_K_M", "Q5_K_S", "Q6_K", "Q8_0", "Q3_K_M"]
SHARD = re.compile(r"-\d{5}-of-\d{5}\.gguf$", re.I)
BUDGET_GB = VRAM_GB * 0.85 - 2.5   # weights must fit GPU0 minus KV cache room


def group_key(f):
    return SHARD.sub("", f) if SHARD.search(f) else f[:-5]


def pick_gguf(files, sizes, quant=None):
    groups = {}
    for f in files:
        groups.setdefault(group_key(f), []).append(f)
    if quant:
        groups = {k: v for k, v in groups.items() if quant.lower() in k.lower()}
        if not groups:
            raise RuntimeError(f"no gguf matching '{quant}'. have: {sorted({group_key(f) for f in files})}")

    def total(k):
        return sum(sizes[x] for x in groups[k])

    def rank(k):
        up = k.upper()
        for i, q in enumerate(QUANT_PREF):
            if q in up:
                return (i, total(k))
        return (99, total(k))

    key = min(groups, key=rank)
    return key, sorted(groups[key])


def resolve(src):
    """returns (kind, path, alias, repo). kind = gguf | hfdir"""
    if os.path.exists(src):
        p = Path(src)
        if p.is_file():
            if p.suffix == ".gguf":
                return "gguf", str(p), p.stem, None
            raise RuntimeError(f"'{p.suffix}' file not loadable (.pth is a raw torch checkpoint). Need .gguf or an HF folder with safetensors.")
        gg = sorted(x for x in p.rglob("*.gguf") if "mmproj" not in x.name.lower())
        if gg:
            return "gguf", str(gg[0]), gg[0].stem, None
        if list(p.glob("*.safetensors")) or list(p.glob("pytorch_model*.bin")):
            return "hfdir", str(p), str(p), None
        raise RuntimeError("folder has no .gguf / safetensors / pytorch_model*.bin")

    quant = fname = None
    if src.endswith(".gguf"):
        parts = src.split("/")
        repo, fname = "/".join(parts[:2]), "/".join(parts[2:])
    elif ":" in src:
        repo, quant = src.split(":", 1)
    else:
        repo = src

    info = HfApi().model_info(repo, files_metadata=True)
    sizes = {s.rfilename: (s.size or 0) for s in info.siblings}
    ggufs = [f for f in sizes if f.lower().endswith(".gguf") and "mmproj" not in f.lower()]
    if fname:
        if fname not in sizes:
            raise RuntimeError(f"{fname} not in {repo}. gguf files: {ggufs}")
        ggufs = [f for f in ggufs if group_key(f) == group_key(fname)]

    if ggufs:
        key, parts = pick_gguf(ggufs, sizes, quant)
        gb = sum(sizes[x] for x in parts) / 1e9
        print(f"gguf: {parts[0]}  ({len(parts)} file, {gb:.1f} GB, budget {BUDGET_GB:.1f} GB)")
        if gb > BUDGET_GB:
            raise RuntimeError(f"{gb:.1f} GB > budget {BUDGET_GB:.1f} GB. Use a smaller quant: SRC='{repo}:Q4_K_M' or another model.")
        if len(parts) == 1:
            path = hf_hub_download(repo, parts[0])
        else:
            snap = snapshot_download(repo, allow_patterns=parts)
            path = str(Path(snap) / parts[0])
        return "gguf", path, repo, repo

    if any(f.endswith(".safetensors") or re.search(r"pytorch_model.*\.bin$", f) for f in sizes):
        gb = sum(v for k, v in sizes.items() if k.endswith(".safetensors")) / 1e9
        print(f"no gguf in {repo}. HF weights ~{gb:.1f} GB. mode: {NON_GGUF}")
        snap = snapshot_download(repo, ignore_patterns=["*.gguf", "*.pth", "original/*", "*.onnx", "*.msgpack", "*.h5"])
        return "hfdir", snap, repo, repo

    raise RuntimeError(f"{repo}: no .gguf, no safetensors, no pytorch_model*.bin. A repo with only .pth cannot be served. Find a GGUF or HF-format repo.")


KIND, PATH, ALIAS, REPO = resolve(SRC)

if KIND == "hfdir" and NON_GGUF == "convert":
    if not LLAMA.exists():
        sh(f"git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA}")
    sh(f"{sys.executable} -m pip install -q {LLAMA}/gguf-py sentencepiece protobuf")
    out = f"/kaggle/working/{ALIAS.replace('/', '_')}-q8_0.gguf"
    if not os.path.exists(out):
        print("converting to GGUF q8_0 (needs RAM ~ model size, few min)...")
        sh(f"{sys.executable} {LLAMA}/convert_hf_to_gguf.py {PATH} --outfile {out} --outtype q8_0", log="convert.log")
    gb = os.path.getsize(out) / 1e9
    print(f"converted: {out} ({gb:.1f} GB)")
    if gb > BUDGET_GB:
        raise RuntimeError(f"q8_0 {gb:.1f} GB > budget {BUDGET_GB:.1f} GB. Use smaller model or NON_GGUF='transformers'.")
    KIND, PATH = "gguf", out

print("KIND:", KIND, "| PATH:", PATH, "| ALIAS:", ALIAS)


gguf: Nemotron-3.5-Bolt-E4B-Q4_K_M.gguf  (1 file, 6.2 GB, budget 10.2 GB)


Nemotron-3.5-Bolt-E4B-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 6.18GB            

Nemotron-3.5-Bolt-E4B-Q4_K_M.gguf: downloading bytes:           |  0.00B            

KIND: gguf | PATH: /root/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3.5-Bolt-E4B-4GB/snapshots/1db54d5b12533ff1bcaa8e42cf4c667988a1f1be/Nemotron-3.5-Bolt-E4B-Q4_K_M.gguf | ALIAS: nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB


In [5]:
def wait_ready(p, url, tries=180):
    for _ in range(tries):
        if p.poll() is not None:
            return False
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False


LLM_PROC = None
if KIND == "gguf":
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": LLM_GPU,
           "LD_LIBRARY_PATH": ld_path(BIN)}
    base = [str(BIN), "-m", PATH, "--alias", ALIAS, "-ngl", "999", "-c", str(CTX),
            "-np", "1", "--jinja", "--host", "127.0.0.1", "--port", "8000"]
    tries = [["--reasoning-budget", "0"], []] if not THINK else [[]]
    for extra in tries:
        LLM_PROC = subprocess.Popen(base + extra, stdout=open("serve.log", "w"),
                                    stderr=subprocess.STDOUT, env=env)
        if wait_ready(LLM_PROC, "http://127.0.0.1:8000/health"):
            print("llama-server up. extra flags:", extra)
            break
        if LLM_PROC.poll() is None:
            LLM_PROC.kill()
        print("launch failed with", extra, "\n", open("serve.log").read()[-1500:])
        LLM_PROC = None
    assert LLM_PROC, "llama-server did not start. see serve.log"
else:
    SERVE_MODEL = REPO or PATH
    ALIAS = SERVE_MODEL
    sh(f"{sys.executable} -m pip install -q -U 'transformers[serving]' bitsandbytes accelerate")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": LLM_GPU, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    LLM_PROC = subprocess.Popen(
        ["transformers", "serve", "--port", "8000", "--quantization", "bnb-4bit", "--model-timeout", "43200"],
        stdout=open("serve.log", "w"), stderr=subprocess.STDOUT, env=env)
    assert wait_ready(LLM_PROC, "http://127.0.0.1:8000/v1/models", 120), "transformers serve failed. see serve.log"


def chat(text, max_tokens=16):
    t = time.time()
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions",
                      json={"model": ALIAS, "messages": [{"role": "user", "content": text}], "max_tokens": max_tokens},
                      timeout=1800)
    dt = time.time() - t
    if r.status_code != 200:
        raise RuntimeError(r.text[:500] + "\n--- serve.log ---\n" + open("serve.log").read()[-3000:])
    d = r.json()
    tps = d.get("timings", {}).get("predicted_per_second")
    print(f"  200 in {dt:.1f}s" + (f", {tps:.1f} tok/s" if tps else ""), "|", d["choices"][0]["message"].get("content", "")[:60])


print("warmup...")
chat("hi")
print("long prompt (~2500 words)...")
chat("word " * 2500)
print("repeat long prompt (prompt cache should make this fast)...")
chat("word " * 2500)

print("tool-call test...")
r = requests.post("http://127.0.0.1:8000/v1/chat/completions", json={
    "model": ALIAS,
    "messages": [{"role": "user", "content": "What time is it in Tokyo?"}],
    "tools": [{"type": "function", "function": {
        "name": "get_current_time", "description": "Get current time in a timezone.",
        "parameters": {"type": "object", "properties": {"timezone": {"type": "string"}}, "required": ["timezone"]}}}],
    "max_tokens": 300}, timeout=300)
msg = r.json()["choices"][0]["message"]
print("  TOOL CALLS OK" if msg.get("tool_calls") else "  TOOL CALLS FAILED. content:", msg.get("tool_calls") or msg.get("content", "")[:300])

for l in [l for l in open("serve.log") if re.search(r"offloaded|CUDA0|CUDA_Host|KV", l)][:6]:
    print(" ", l.strip()[:160])
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv


llama-server up. extra flags: ['--reasoning-budget', '0']
warmup...
  200 in 47.9s, 34.9 tok/s | Hi! How can I help you today? 😊
long prompt (~2500 words)...
  200 in 29.5s, 47.8 tok/s | The user has provided a sequence of the word "word" repeated
repeat long prompt (prompt cache should make this fast)...
  200 in 0.3s, 48.4 tok/s | The user has provided a very long string consisting only of 
tool-call test...
  TOOL CALLS OK [{'type': 'function', 'function': {'name': 'get_current_time', 'arguments': '{"timezone":"Asia/Tokyo"}'}, 'id': 'lRg4fSfgsuHK8f11RSeCnBJ5K8rKy8AT'}]
index, memory.used [MiB], memory.total [MiB]
0, 3695 MiB, 15360 MiB
1, 0 MiB, 15360 MiB


In [6]:
if not os.path.exists("cloudflared"):
    sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")


def tunnel(port, log):
    subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"], stderr=open(log, "w"))
    for _ in range(40):
        time.sleep(1)
        mm = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(log).read())
        if mm:
            return mm.group(0)
    raise RuntimeError("no tunnel url:\n" + open(log).read()[-1500:])


LLM_URL = tunnel(8000, "cf.log") + "/v1"
print("BASE URL:", LLM_URL)


BASE URL: https://experiments-ray-dogs-advertising.trycloudflare.com/v1


In [7]:
%%writefile embed_server.py
import base64, io, os, tempfile, threading, uuid
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
from sentence_transformers import SentenceTransformer

app = FastAPI()
m = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
m.max_seq_length = 256

OCR_ID = os.environ.get("OCR_MODEL", "zai-org/GLM-OCR")
MAX_SIDE = int(os.environ.get("OCR_MAX_SIDE", "1024"))
_ocr = {}
_lock = threading.Lock()
_jobs = {}


def ocr_model():
    if not _ocr:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        _ocr["p"] = AutoProcessor.from_pretrained(OCR_ID)
        _ocr["m"] = AutoModelForImageTextToText.from_pretrained(
            OCR_ID, torch_dtype="auto", device_map="auto")
    return _ocr["p"], _ocr["m"]


class Req(BaseModel):
    model: str | None = None
    input: str | list[str]


class OcrReq(BaseModel):
    image: str
    prompt: str = "Text Recognition:"
    max_new_tokens: int = 1024


@app.post("/v1/embeddings")
def emb(r: Req):
    xs = [r.input] if isinstance(r.input, str) else r.input
    v = m.encode(xs, normalize_embeddings=True, batch_size=16)
    return {"object": "list", "model": "bge-small-en-v1.5",
            "data": [{"object": "embedding", "index": i, "embedding": e.tolist()} for i, e in enumerate(v)],
            "usage": {"prompt_tokens": 0, "total_tokens": 0}}


def _set(jid, v):
    _jobs[jid] = v
    while len(_jobs) > 50:
        _jobs.pop(next(iter(_jobs)))


def _run(jid, r):
    path = None
    try:
        img = Image.open(io.BytesIO(base64.b64decode(r.image))).convert("RGB")
        img.thumbnail((MAX_SIDE, MAX_SIDE))
        with _lock:
            p, mod = ocr_model()
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
                img.save(f, "PNG")
                path = f.name
            msgs = [{"role": "user", "content": [
                {"type": "image", "url": path},
                {"type": "text", "text": r.prompt}]}]
            inputs = p.apply_chat_template(
                msgs, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt").to(mod.device)
            inputs.pop("token_type_ids", None)
            with torch.inference_mode():
                out = mod.generate(**inputs, max_new_tokens=r.max_new_tokens, do_sample=False)
            text = p.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        _set(jid, {"status": "done", "text": text.strip()})
    except Exception as e:
        _set(jid, {"status": "error", "error": f"{type(e).__name__}: {e}"})
    finally:
        if path and os.path.exists(path):
            os.unlink(path)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


@app.post("/v1/ocr/start")
def ocr_start(r: OcrReq):
    jid = uuid.uuid4().hex
    _set(jid, {"status": "running"})
    threading.Thread(target=_run, args=(jid, r), daemon=True).start()
    return {"id": jid}


@app.get("/v1/ocr/{jid}")
def ocr_poll(jid: str):
    return _jobs.get(jid, {"status": "missing"})


def _preload():
    with _lock:
        ocr_model()


threading.Thread(target=_preload, daemon=True).start()


Writing embed_server.py


In [8]:
import base64, io
from PIL import Image, ImageDraw, ImageFont

env = {**os.environ, "CUDA_VISIBLE_DEVICES": OCR_GPU, "OCR_MAX_SIDE": str(OCR_SIDE),
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
EMB_PROC = subprocess.Popen([sys.executable, "-m", "uvicorn", "embed_server:app", "--port", "8001"],
                            stdout=open("embed.log", "w"), stderr=subprocess.STDOUT, env=env)
assert wait_ready(EMB_PROC, "http://127.0.0.1:8001/docs", 120), "embed server failed:\n" + open("embed.log").read()[-2000:]
EMBED_URL = tunnel(8001, "cf2.log") + "/v1"
print("EMBED URL:", EMBED_URL)

img = Image.new("RGB", (700, 140), "white")
ImageDraw.Draw(img).text((20, 40), "OCR TEST 12345", fill="black", font=ImageFont.load_default(size=48))
buf = io.BytesIO()
img.save(buf, "PNG")
r = requests.post("http://127.0.0.1:8001/v1/ocr/start",
                  json={"image": base64.b64encode(buf.getvalue()).decode()}, timeout=30)
jid = r.json()["id"]
d = {}
for _ in range(200):
    time.sleep(3)
    d = requests.get(f"http://127.0.0.1:8001/v1/ocr/{jid}", timeout=30).json()
    if d["status"] != "running":
        break
print("OCR sanity:", d)

print("\n===== paste into local .env =====")
print(f"BRAIN_LLM_BASE_URL={LLM_URL}")
print(f"BRAIN_LLM_NAME={ALIAS}")
print(f"EMBED_BASE_URL={EMBED_URL}")


EMBED URL: https://leads-enlargement-turtle-alfred.trycloudflare.com/v1
OCR sanity: {'status': 'done', 'text': 'OCR TEST 12345'}

===== paste into local .env =====
BRAIN_LLM_BASE_URL=https://experiments-ray-dogs-advertising.trycloudflare.com/v1
BRAIN_LLM_NAME=nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB
EMBED_BASE_URL=https://leads-enlargement-turtle-alfred.trycloudflare.com/v1


In [9]:
!nvidia-smi -L
!pip install -q faster-whisper silero-vad fastapi "uvicorn[standard]"

GPU 0: Tesla T4 (UUID: GPU-154c331f-83b7-1cb0-dc87-bf8598a8c93f)
GPU 1: Tesla T4 (UUID: GPU-9a05e0e4-5412-22b7-33d5-d01f01e0dca0)


In [10]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
import os, nvidia.cublas, nvidia.cudnn
libs = [p + "/lib" for m in (nvidia.cublas, nvidia.cudnn) for p in m.__path__]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(os.environ["LD_LIBRARY_PATH"])

/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64


In [11]:
%%writefile stt_server.py
import asyncio, json, os, threading, time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from fastapi import FastAPI, Query, WebSocket
from faster_whisper import WhisperModel

MODEL = os.environ.get("STT_MODEL", "large-v3-turbo")   # or distil-large-v3 / small.en
LANG = os.environ.get("STT_LANG", "en") or None         # "" = auto detect (slower)
TOKEN = os.environ.get("STT_TOKEN", "")
PROMPT = os.environ.get("STT_PROMPT") or None           # vocab hint: "Ultron, Kolkata, ..."
DEVICE = os.environ.get("STT_DEVICE", "cuda")
COMPUTE = os.environ.get("STT_COMPUTE", "float16" if DEVICE == "cuda" else "int8")
BEAM = int(os.environ.get("STT_BEAM", "1"))
END_MS = int(os.environ.get("STT_END_MS", "600"))        # silence that ends an utterance
PARTIAL_S = float(os.environ.get("STT_PARTIAL_S", "0.7"))
MIN_RMS = float(os.environ.get("STT_MIN_RMS", "0.010"))
VAD = os.environ.get("STT_VAD", "silero")                 # "energy" = old loudness threshold
VAD_ON = float(os.environ.get("STT_VAD_ON", "0.5"))       # speech starts above this prob
VAD_OFF = float(os.environ.get("STT_VAD_OFF", "0.35"))    # speech continues above this prob
MAX_UTT_S = 25
PREROLL_MS = 500
SR = 16000
JUNK = {"you", "thank you.", "thanks for watching!", "thank you for watching.", "bye.", "."}

print(f"loading {MODEL} on {DEVICE}/{COMPUTE}", flush=True)
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE)
lock = threading.Lock()
pool = ThreadPoolExecutor(max_workers=1)


def run(audio: np.ndarray, beam: int = BEAM) -> str:
    with lock:
        segs, _ = model.transcribe(
            audio, language=LANG, beam_size=beam, temperature=0.0,
            condition_on_previous_text=False, without_timestamps=True,
            vad_filter=True, vad_parameters={"min_silence_duration_ms": 300, "speech_pad_ms": 150},
            initial_prompt=PROMPT, no_speech_threshold=0.6)
        out = []
        for s in segs:                                   # generator: consume inside lock
            if s.no_speech_prob > 0.6 and s.avg_logprob < -1.0:
                continue
            out.append(s.text.strip())
    return " ".join(out).strip()


run(np.zeros(SR, dtype=np.float32))                      # warm up kernels
print("model ready", flush=True)

load_vad = None
if VAD == "silero":
    try:
        import torch
        from silero_vad import load_silero_vad as load_vad
        load_vad()
        print("silero vad ready", flush=True)
    except Exception as e:
        load_vad = None
        print("silero unavailable, using energy vad:", type(e).__name__, e, flush=True)

app = FastAPI()


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL}


@app.websocket("/ws")
async def ws_endpoint(ws: WebSocket, token: str = Query("")):
    if TOKEN and token != TOKEN:
        await ws.close(code=4401)
        return
    await ws.accept()
    loop = asyncio.get_running_loop()
    vad = load_vad() if load_vad else None     # own model per connection (LSTM state)
    rest = np.zeros(0, np.float32)             # leftover samples, silero wants 512-sample frames
    pre, buf = [], []
    st = {"speaking": False, "silence": 0.0, "n": 0, "nf": 0.003, "last_p": 0.0, "busy": False, "uid": 0, "p": 0.0}

    async def send(o):
        try:
            await ws.send_text(json.dumps(o))
        except Exception:
            pass

    async def partial(audio, uid):
        try:
            text = await loop.run_in_executor(pool, run, audio, 1)
            if text and st["speaking"] and uid == st["uid"]:
                await send({"type": "partial", "text": text})
        finally:
            st["busy"] = False

    def speech_prob(x):
        nonlocal rest
        a = np.concatenate([rest, x])
        n = len(a) // 512
        rest = a[n * 512:]
        if n:
            with torch.no_grad():
                st["p"] = max(float(vad(torch.from_numpy(a[i * 512:(i + 1) * 512]), SR)) for i in range(n))
        return st["p"]

    def clear():
        if vad is not None:
            vad.reset_states()
        buf.clear()
        st.update(speaking=False, silence=0.0, n=0)
        st["uid"] += 1

    async def finalize():
        audio = np.concatenate(buf) if buf else np.zeros(0, np.float32)
        clear()
        await send({"type": "vad", "speaking": False})
        if len(audio) < 0.35 * SR:
            await send({"type": "final", "text": "", "ms": 0})
            return
        t0 = time.time()
        text = await loop.run_in_executor(pool, run, audio, BEAM)
        if text.lower() in JUNK and len(audio) < 1.2 * SR:
            text = ""
        await send({"type": "final", "text": text, "ms": int((time.time() - t0) * 1000)})

    try:
        while True:
            m = await ws.receive()
            if m["type"] == "websocket.disconnect":
                break
            if m.get("text"):
                j = json.loads(m["text"])
                if j.get("type") == "end" and st["speaking"]:
                    await finalize()
                elif j.get("type") == "end":
                    await send({"type": "final", "text": "", "ms": 0})
                elif j.get("type") == "reset":
                    clear(); pre.clear()
                continue
            b = m.get("bytes")
            if not b:
                continue
            x = np.frombuffer(b, dtype=np.int16).astype(np.float32) / 32768.0
            ms = len(x) / SR * 1000
            rms = float(np.sqrt(np.mean(x * x)) + 1e-9)
            if vad is not None:
                voiced = speech_prob(x) >= (VAD_OFF if st["speaking"] else VAD_ON)
            else:
                voiced = rms > max(MIN_RMS, st["nf"] * 3.0)

            if not st["speaking"]:
                st["nf"] = 0.95 * st["nf"] + 0.05 * min(rms, 0.05)
                pre.append(x)
                while sum(len(p) for p in pre) > PREROLL_MS / 1000 * SR:
                    pre.pop(0)
                if voiced:
                    buf[:] = pre; pre.clear()
                    st.update(speaking=True, silence=0.0, n=sum(len(p) for p in buf), last_p=time.time())
                    await send({"type": "vad", "speaking": True})
            else:
                buf.append(x); st["n"] += len(x)
                st["silence"] = 0.0 if voiced else st["silence"] + ms
                if st["silence"] >= END_MS or st["n"] >= MAX_UTT_S * SR:
                    await finalize()
                    continue
                if not st["busy"] and time.time() - st["last_p"] >= PARTIAL_S and st["n"] > 0.5 * SR:
                    st["busy"] = True; st["last_p"] = time.time()
                    asyncio.create_task(partial(np.concatenate(buf)[-15 * SR:], st["uid"]))
    except Exception as e:
        print("ws error:", type(e).__name__, e, flush=True)

Writing stt_server.py


In [12]:
import os, re, secrets, subprocess, time, requests

# ---------- EDIT HERE ----------
os.environ["STT_MODEL"] = "large-v3-turbo"   # "distil-large-v3" (English, faster) | "small.en" (tiny)
os.environ["STT_LANG"] = "en"                # "" = auto detect (slower)
os.environ["STT_END_MS"] = "600"             # silence before utterance ends. 400 = snappier, cuts pauses
os.environ["STT_PARTIAL_S"] = "0.7"          # live partial every N sec
os.environ["STT_PROMPT"] = "Ultron"          # vocab hint, comma separated words
os.environ["STT_VAD"] = "silero"             # "energy" = old loudness VAD
os.environ["STT_VAD_ON"] = "0.5"             # speech start prob. higher = fewer false triggers
os.environ["STT_VAD_OFF"] = "0.35"           # keep-speaking prob. lower = holds through pauses
# -------------------------------

os.environ["STT_TOKEN"] = secrets.token_urlsafe(16)

def wait_ready(p, url, tries=180):
    for _ in range(tries):
        if p.poll() is not None:
            return False
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False

SRV = subprocess.Popen(["python", "-m", "uvicorn", "stt_server:app", "--host", "127.0.0.1", "--port", "8002"],
                       stdout=open("stt.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
assert wait_ready(SRV, "http://127.0.0.1:8002/health"), "server failed:\n" + open("stt.log").read()[-2500:]

if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8002"], stderr=open("cf_stt.log", "w"))
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf_stt.log").read())
    if m:
        url = m.group(0); break
assert url, "no tunnel url:\n" + open("cf_stt.log").read()[-1500:]

print("===== paste into local .env =====")
print("STT_WS_URL=" + url.replace("https://", "wss://") + "/ws")
print("STT_TOKEN=" + os.environ["STT_TOKEN"])

===== paste into local .env =====
STT_WS_URL=wss://cork-pipes-ata-businesses.trycloudflare.com/ws
STT_TOKEN=J-DWa-vAvxdXUeRPWlvIwA


In [13]:
print("\n===== paste into local .env =====")
print(f"BRAIN_LLM_BASE_URL={LLM_URL}")
print(f"BRAIN_LLM_NAME={ALIAS}")
print(f"EMBED_BASE_URL={EMBED_URL}")
print("STT_WS_URL=" + url.replace("https://", "wss://") + "/ws")
print("STT_TOKEN=" + os.environ["STT_TOKEN"])


===== paste into local .env =====
BRAIN_LLM_BASE_URL=https://experiments-ray-dogs-advertising.trycloudflare.com/v1
BRAIN_LLM_NAME=nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB
EMBED_BASE_URL=https://leads-enlargement-turtle-alfred.trycloudflare.com/v1
STT_WS_URL=wss://cork-pipes-ata-businesses.trycloudflare.com/ws
STT_TOKEN=J-DWa-vAvxdXUeRPWlvIwA
